# Superstore EDA
NumPy, Pandas, Matplotlib only.

## 1. Imports & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Sample_-_Superstore.csv", encoding="latin1")
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"]  = pd.to_datetime(df["Ship Date"])
df["Year"]       = df["Order Date"].dt.year
df["Month"]      = df["Order Date"].dt.month

print("Rows   :", len(df))
print("Columns:", len(df.columns))
df.head()

## 2. Data Types & Missing Values

In [ ]:
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Duplicates:", df.duplicated().sum())

## 3. Basic Statistics

In [ ]:
print(df[["Sales", "Profit", "Quantity", "Discount"]].describe())

In [ ]:
for col in ["Sales", "Profit", "Quantity", "Discount"]:
    vals = df[col].values
    print(col)
    print("  Mean  :", round(np.mean(vals), 2))
    print("  Median:", round(np.median(vals), 2))
    print("  Std   :", round(np.std(vals), 2))
    print("  Min   :", round(np.min(vals), 2))
    print("  Max   :", round(np.max(vals), 2))
    print()

## 4. Key Business Numbers

In [ ]:
total_sales  = df["Sales"].sum()
total_profit = df["Profit"].sum()
margin       = (total_profit / total_sales) * 100
print("Total Sales     : $", round(total_sales, 2))
print("Total Profit    : $", round(total_profit, 2))
print("Profit Margin   :", round(margin, 2), "%")
print("Unique Orders   :", df["Order ID"].nunique())
print("Unique Customers:", df["Customer ID"].nunique())
print("Date Range      :", df["Order Date"].min().date(), "to", df["Order Date"].max().date())

## 5. Category Analysis

In [ ]:
cat = df.groupby("Category")[["Sales", "Profit"]].sum()
print(cat)

x = np.arange(len(cat))
width = 0.4
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, cat["Sales"].values,  width, label="Sales",  color="steelblue")
ax.bar(x + width/2, cat["Profit"].values, width, label="Profit", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(cat.index)
ax.set_title("Sales and Profit by Category")
ax.set_ylabel("Amount ($)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Sub-Category Profit

In [ ]:
sub_profit = df.groupby("Sub-Category")["Profit"].sum().sort_values()
print(sub_profit)

colors = ["crimson" if v < 0 else "steelblue" for v in sub_profit.values]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(sub_profit.index, sub_profit.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Profit by Sub-Category (red = loss)")
ax.set_xlabel("Profit ($)")
plt.tight_layout()
plt.show()

## 7. Region Analysis

In [ ]:
reg = df.groupby("Region")[["Sales", "Profit"]].sum()
print(reg)

x = np.arange(len(reg))
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, reg["Sales"].values,  width, label="Sales",  color="steelblue")
ax.bar(x + width/2, reg["Profit"].values, width, label="Profit", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(reg.index)
ax.set_title("Sales and Profit by Region")
ax.set_ylabel("Amount ($)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Segment Breakdown

In [ ]:
seg = df.groupby("Segment")["Sales"].sum()
print(seg)

fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(seg.values, labels=seg.index, autopct="%1.1f%%", colors=["steelblue", "seagreen", "coral"])
ax.set_title("Sales by Segment")
plt.show()

## 9. Yearly Sales Trend

In [ ]:
year = df.groupby("Year")[["Sales", "Profit"]].sum()
print(year)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(year.index, year["Sales"].values,  marker="o", label="Sales",  color="steelblue")
ax.plot(year.index, year["Profit"].values, marker="s", label="Profit", color="seagreen", linestyle="--")
ax.set_title("Sales and Profit by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Amount ($)")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Monthly Seasonality

In [ ]:
month_avg = df.groupby("Month")["Sales"].mean()
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(month_labels, month_avg.values, color="steelblue")
ax.set_title("Average Monthly Sales")
ax.set_ylabel("Avg Sales ($)")
plt.tight_layout()
plt.show()

## 11. Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df["Sales"],  bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Sales Distribution")
axes[0].set_xlabel("Sales ($)")
axes[1].hist(df["Profit"], bins=50, color="seagreen",  edgecolor="white")
axes[1].set_title("Profit Distribution")
axes[1].set_xlabel("Profit ($)")
plt.tight_layout()
plt.show()

## 12. Boxplots

In [ ]:
cols = ["Sales", "Profit", "Quantity", "Discount"]
box_colors = ["steelblue", "seagreen", "coral", "orchid"]
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for i in range(4):
    axes[i].boxplot(df[cols[i]], patch_artist=True, boxprops=dict(facecolor=box_colors[i], alpha=0.7))
    axes[i].set_title(cols[i])
    axes[i].set_xticks([])
plt.suptitle("Boxplots")
plt.tight_layout()
plt.show()

## 13. Outlier Detection (IQR)

In [ ]:
for col in ["Sales", "Profit"]:
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n     = len(df[(df[col] < lower) | (df[col] > upper)])
    print(col)
    print("  Fence   :", round(lower, 2), "to", round(upper, 2))
    print("  Outliers:", n, "(", round(n / len(df) * 100, 1), "%)")
    print()

## 14. Correlation Heatmap

In [ ]:
num_cols  = ["Sales", "Profit", "Quantity", "Discount"]
corr      = df[num_cols].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(num_cols)
ax.set_yticklabels(num_cols)
for i in range(4):
    for j in range(4):
        ax.text(j, i, round(corr.values[i, j], 2), ha="center", va="center", fontsize=11)
plt.colorbar(im, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 15. Discount vs Profit

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df["Discount"], df["Profit"], alpha=0.3, color="steelblue", s=15)
ax.axhline(0, color="red", linewidth=0.9, linestyle="--")
ax.set_title("Discount vs Profit")
ax.set_xlabel("Discount")
ax.set_ylabel("Profit ($)")
plt.tight_layout()
plt.show()

## 16. Avg Profit by Discount Bucket

In [ ]:
bins    = [-0.01, 0, 0.2, 0.4, 0.6, 0.9]
blabels = ["0%", "1-20%", "21-40%", "41-60%", "61-90%"]
df["Discount Bin"] = pd.cut(df["Discount"], bins=bins, labels=blabels)
disc_profit = df.groupby("Discount Bin", observed=True)["Profit"].mean()
print(disc_profit.round(2))

colors = ["crimson" if v < 0 else "seagreen" for v in disc_profit.values]
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(disc_profit.index, disc_profit.values, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Average Profit by Discount Level")
ax.set_xlabel("Discount Range")
ax.set_ylabel("Avg Profit ($)")
plt.tight_layout()
plt.show()

## 17. Top 10 States by Sales

In [ ]:
state_sales = df.groupby("State")["Sales"].sum().sort_values(ascending=False).head(10)
print(state_sales)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(state_sales.index[::-1], state_sales.values[::-1], color="steelblue")
ax.set_title("Top 10 States by Sales")
ax.set_xlabel("Sales ($)")
plt.tight_layout()
plt.show()

## 18. Key Findings

In [ ]:
print("KEY FINDINGS")
print("=" * 50)
print("1. No missing values. 9,994 rows, 21 columns.")
print("2. Total sales .30M | Profit 86K | Margin 12.5%.")
print("3. Technology has the best margin (~17%).")
print("4. Furniture barely breaks even (~2.5% margin).")
print("5. Tables, Bookcases, Supplies make a loss.")
print("6. Discounts above 20% flip profit negative.")
print("7. West region leads on sales and profit.")
print("8. Sales grew 51% from 2014 to 2017.")
print("9. Nov-Dec is the strongest sales period.")